# ♟️ Benchmark 1 — Précision de l'évaluation des réseaux P.A.W.N.

On compare les **4 réseaux** d'évaluation (sans recherche) à **Stockfish** (profondeur ≥ 20) sur des positions
qu'**aucun modèle n'a vues à l'entraînement** :

| Réseau | Dossier | Taille | Données d'entraînement |
|---|---|---|---|
| PAWN | `Pawn/` | 8×96, 1,41 M | 30 M premières positions |
| PAWN_BIG | `PawnBig-V1/` | 12×128, 3,64 M | 60 M premières positions |
| CONTROLE | `PawnBig-controlleur/` | 12×128, 3,64 M | PAWN big + 30 M suivantes (sans Soluce) |
| SPAWN (= réseau de PWN) | `PWN-soluce/` | 12×128 + Soluce, 3,65 M | PAWN big + 30 M suivantes (avec Soluce) |

Les modèles ont été entraînés sur les **90 M premières lignes** de `lichess/chess-evaluations` : on prend donc les positions
de test dans les **derniers fichiers parquet** du dataset (le notebook vérifie qu'ils commencent après la ligne 90 M).

**Métriques**
- **BCE** : la même perte que pendant l'entraînement (cible `sigmoid(cp/400)`) → comparable aux pertes de validation
- **MAE win-prob** : erreur moyenne sur la probabilité de gain (en points de %)
- **MAE cp** : erreur en centipions (évaluations bornées à ±1000)
- **Spearman** : est-ce que le modèle classe les positions dans le même ordre que Stockfish ?
- **Bon camp** : le modèle dit-il le bon camp gagnant quand |éval Stockfish| ≥ 100 cp ?
- **Accord coup** : à profondeur 1, le coup choisi par le réseau est-il le meilleur coup de Stockfish ? (top-1 / top-3)
- **Symétrie** : une position et son miroir gauche↔droite (sans roque) devraient avoir la même éval
- **Vitesse** : latence (1 position) et débit (positions/s) CPU et GPU

⚙️ Réglages Kaggle : **Internet ON**, accélérateur **GPU T4** (facultatif mais plus rapide).

In [ ]:
# onnxruntime-gpu remplace onnxruntime (les deux ne cohabitent pas) ; marche aussi sans GPU.
# Version fixée : les plus récentes demandent CUDA 13, Kaggle a CUDA 12. [cuda,cudnn] installe les bibliothèques CUDA 12.
!pip uninstall -y -q onnxruntime onnxruntime-gpu > /dev/null 2>&1
!pip install -q chess zstandard "onnxruntime-gpu[cuda,cudnn]==1.24.1"

In [ ]:
%%writefile commun_echecs.py
"""Code commun aux notebooks de benchmark échecs (PAWN, PAWN big, contrôle, SPAWN/PWN).

Tout vient de pawn_app.py : même encodage 69 octets, même recherche « arbre complet »,
même alpha-bêta + quiescence. Seule différence : pas de bruit ni de coups au hasard,
on veut mesurer la force réelle.
"""
import glob
import os
import shutil
import stat
import subprocess
import tarfile
import time
import urllib.request

import chess
import chess.engine
import chess.polyglot
import numpy as np
import onnxruntime as ort

DEPOT = "LiamLitle/ia-de-liam"
BRANCHE = "main"
LFS = f"https://media.githubusercontent.com/media/{DEPOT}/{BRANCHE}/"
DOSSIER_POIDS = os.environ.get("PAWN_POIDS", "/kaggle/working/poids" if os.path.isdir("/kaggle") else "poids")
MATE = 100_000

# nom -> chemin dans le dépôt
RESEAUX = {
    "PAWN": "Pawn/pawn.onnx",
    "PAWN_BIG": "PawnBig-V1/pawn_big.onnx",
    "CONTROLE": "PawnBig-controlleur/pawn_big_controle.onnx",
    "SPAWN": "PWN-soluce/pawn_soluce.onnx",
}


def poids(chemin_depot):
    """cherche le fichier dans /kaggle/input (dataset ajouté au notebook), sinon le télécharge depuis GitHub LFS"""
    nom = os.path.basename(chemin_depot)
    for racine in ("/kaggle/input", DOSSIER_POIDS):
        trouves = glob.glob(os.path.join(racine, "**", nom), recursive=True)
        trouves = [t for t in trouves if os.path.getsize(t) > 1000]  # pas un pointeur LFS
        if trouves:
            return trouves[0]
    os.makedirs(DOSSIER_POIDS, exist_ok=True)
    dest = os.path.join(DOSSIER_POIDS, nom)
    print(f"téléchargement de {chemin_depot} ...")
    urllib.request.urlretrieve(LFS + chemin_depot, dest)
    if os.path.getsize(dest) < 1000:
        raise RuntimeError(f"{dest} ressemble à un pointeur LFS, pas au vrai fichier")
    return dest


_CUDA_OK = None  # None = pas encore essayé, False = échec -> on reste sur CPU sans réessayer


def providers(gpu=True):
    dispo = ort.get_available_providers()
    if gpu and _CUDA_OK is not False and "CUDAExecutionProvider" in dispo:
        try:
            ort.preload_dlls()  # charge CUDA/cuDNN depuis les paquets pip nvidia-* (déjà là avec torch sur Kaggle)
        except Exception:
            pass
        return ["CUDAExecutionProvider", "CPUExecutionProvider"]
    return ["CPUExecutionProvider"]


PIECES_ID = {c: i + 1 for i, c in enumerate("PNBRQK")}


def enc_fen(fen):
    f = fen.split(" ")
    noir = f[1] == "b"
    out = bytearray(69)
    r = c = 0
    for ch in f[0]:
        if ch == "/":
            r += 1
            c = 0
        elif ch.isdigit():
            c += int(ch)
        else:
            nous = ch.isupper() != noir
            out[(r if noir else 7 - r) * 8 + c] = PIECES_ID[ch.upper()] + (0 if nous else 6)
            c += 1
    ours, theirs = ("kq", "KQ") if noir else ("KQ", "kq")
    out[64] = ours[0] in f[2]
    out[65] = ours[1] in f[2]
    out[66] = theirs[0] in f[2]
    out[67] = theirs[1] in f[2]
    if f[3] != "-":
        out[68] = ord(f[3][0]) - 96
    return bytes(out)


class Evaluateur:
    """un réseau ONNX : liste de FEN -> centipions du point de vue du camp au trait"""

    def __init__(self, nom, gpu=True, threads=0):
        self.nom = nom
        so = ort.SessionOptions()
        if threads:
            so.intra_op_num_threads = threads
            so.inter_op_num_threads = 1
        global _CUDA_OK
        prov = providers(gpu)
        self.session = ort.InferenceSession(poids(RESEAUX[nom]), so, providers=prov)
        if "CUDAExecutionProvider" in prov:
            _CUDA_OK = "CUDAExecutionProvider" in self.session.get_providers()
            if not _CUDA_OK:
                print("⚠️ GPU indisponible pour onnxruntime : on continue sur CPU (plus lent mais résultats identiques)")
                ort.set_default_logger_severity(4)
        self.nb_evals = 0
        self.cache = {}

    def evals(self, fens, lot=4096):
        if not fens:
            return np.zeros(0, dtype=np.float32)
        res = []
        for i in range(0, len(fens), lot):
            x = np.frombuffer(b"".join(map(enc_fen, fens[i:i + lot])), np.uint8).reshape(-1, 69).copy()
            res.append(self.session.run(None, {"board": x})[0])
        self.nb_evals += len(fens)
        return np.concatenate(res)

    def eval1(self, fen):
        # l'alpha-bêta réévalue souvent les mêmes positions en quiescence : le réseau est
        # déterministe, donc un cache ne change rien au résultat, juste la vitesse
        v = self.cache.get(fen)
        if v is None:
            if len(self.cache) > 2_000_000:
                self.cache.clear()
            v = self.cache[fen] = float(self.evals([fen])[0])
        return v


# -- recherche « arbre complet » (PAWN, PAWN big, contrôle, SPAWN) -----------------

def arbre(board, depth, ply, feuilles):
    if not any(board.legal_moves):
        return ("t", -(MATE - ply) if board.is_check() else 0)
    if ply > 0 and (board.is_insufficient_material() or board.is_repetition(2) or board.halfmove_clock >= 100):
        return ("t", 0)
    if depth == 0:
        feuilles.append(board.fen())
        return ("f", len(feuilles) - 1)
    fils = []
    for mv in list(board.legal_moves):
        board.push(mv)
        fils.append((mv, arbre(board, depth - 1, ply + 1, feuilles)))
        board.pop()
    return ("n", fils)


def valeur(noeud, v):
    genre, x = noeud
    if genre == "t":
        return x
    if genre == "f":
        return v[x]
    return max(-valeur(c, v) for _, c in x)


def choisir_arbre(board, ev, depth):
    feuilles = []
    racine = arbre(board, depth, 0, feuilles)
    v = ev.evals(feuilles) if feuilles else []
    notes = [(-valeur(c, v), mv) for mv, c in racine[1]]
    return max(notes, key=lambda t: t[0])[1]


# -- recherche alpha-bêta + quiescence (PWN) --------------------------------------

VAL_PIECE = {chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3, chess.ROOK: 5, chess.QUEEN: 9, chess.KING: 0}


def trier_coups(board, coups, priorite):
    def cle(mv):
        if mv == priorite:
            return 10_000
        if board.is_capture(mv):
            prise = board.piece_type_at(mv.to_square) or chess.PAWN
            attaquant = board.piece_type_at(mv.from_square)
            return 1000 + VAL_PIECE[prise] * 10 - VAL_PIECE[attaquant]
        if board.gives_check(mv):
            return 500
        return 0
    return sorted(coups, key=cle, reverse=True)


def quiescence(board, ev, alpha, beta, prof_q):
    stand_pat = ev.eval1(board.fen())
    if stand_pat >= beta:
        return beta
    alpha = max(alpha, stand_pat)
    if prof_q <= 0:
        return alpha
    captures = [m for m in board.legal_moves if board.is_capture(m)]
    for mv in trier_coups(board, captures, None):
        board.push(mv)
        score = -quiescence(board, ev, -beta, -alpha, prof_q - 1)
        board.pop()
        if score >= beta:
            return beta
        alpha = max(alpha, score)
    return alpha


def negamax(board, ev, profondeur, alpha, beta, ply, prof_q, tt, coup_tt):
    alpha0 = alpha
    if not any(board.legal_moves):
        return -(MATE - ply) if board.is_check() else 0
    if ply > 0 and (board.is_insufficient_material() or board.is_repetition(2) or board.halfmove_clock >= 100):
        return 0
    hash_ = chess.polyglot.zobrist_hash(board)
    entree = tt.get(hash_)
    if entree and entree[0] >= profondeur:
        d, v, drapeau, _ = entree
        if drapeau == "exact":
            return v
        if drapeau == "min":
            alpha = max(alpha, v)
        elif drapeau == "max":
            beta = min(beta, v)
        if alpha >= beta:
            return v
    if profondeur <= 0:
        return quiescence(board, ev, alpha, beta, prof_q)
    meilleur, meilleur_coup = -MATE - 1, None
    for mv in trier_coups(board, list(board.legal_moves), coup_tt.get(hash_)):
        board.push(mv)
        score = -negamax(board, ev, profondeur - 1, -beta, -alpha, ply + 1, prof_q, tt, coup_tt)
        board.pop()
        if score > meilleur:
            meilleur, meilleur_coup = score, mv
        alpha = max(alpha, score)
        if alpha >= beta:
            break
    drapeau = "exact" if alpha0 < meilleur < beta else ("min" if meilleur >= beta else "max")
    tt[hash_] = (profondeur, meilleur, drapeau, meilleur_coup)
    if meilleur_coup is not None:
        coup_tt[hash_] = meilleur_coup
    return meilleur


def choisir_alphabeta(board, ev, depth, prof_q=4):
    tt, coup_tt = {}, {}
    meilleur_coup = None
    for d in range(1, depth + 1):
        alpha, beta = -MATE - 1, MATE + 1
        hash_ = chess.polyglot.zobrist_hash(board)
        meilleur_score, meilleur_du_tour = -MATE - 1, None
        for mv in trier_coups(board, list(board.legal_moves), coup_tt.get(hash_)):
            board.push(mv)
            score = -negamax(board, ev, d - 1, -beta, -alpha, 1, prof_q, tt, coup_tt)
            board.pop()
            if score > meilleur_score:
                meilleur_score, meilleur_du_tour = score, mv
            alpha = max(alpha, score)
        meilleur_coup = meilleur_du_tour
        coup_tt[hash_] = meilleur_coup
    return meilleur_coup


# -- joueurs ------------------------------------------------------------------------
# un « joueur » = un réseau + un algorithme de recherche + une profondeur.
# ex : "PAWN_BIG@arbre2", "SPAWN@arbre1", "PWN@ab3" (PWN = poids SPAWN + alpha-bêta).

def decoder(joueur):
    nom, rech = joueur.split("@")
    if nom == "PWN":
        nom = "SPAWN"
    if rech.startswith("arbre"):
        return nom, "arbre", int(rech[5:])
    if rech.startswith("ab"):
        return nom, "ab", int(rech[2:])
    raise ValueError(joueur)


_EVALS = {}
_SF = {}
GPU = True
THREADS = 0
SF_CHEMIN = None
SF_TEMPS = 0.1


def evaluateur(nom, gpu=None):
    gpu = GPU if gpu is None else gpu
    if (nom, gpu) not in _EVALS:
        _EVALS[(nom, gpu)] = Evaluateur(nom, gpu=gpu, threads=THREADS)
    return _EVALS[(nom, gpu)]


def configurer(gpu=True, threads=0, sf_chemin=None, sf_temps=0.1):
    """à appeler dans chaque processus fils avant de jouer"""
    global GPU, THREADS, SF_CHEMIN, SF_TEMPS
    GPU, THREADS, SF_CHEMIN, SF_TEMPS = gpu, threads, sf_chemin, sf_temps
    _EVALS.clear()
    _SF.clear()


def stockfish(cfg):
    """cfg = "elo1500" (UCI_LimitStrength) ou "skill0" (Skill Level)"""
    if cfg not in _SF:
        e = chess.engine.SimpleEngine.popen_uci(SF_CHEMIN)
        opts = {"Threads": 1, "Hash": 16}
        if cfg.startswith("elo"):
            opts.update({"UCI_LimitStrength": True, "UCI_Elo": int(cfg[3:])})
        elif cfg.startswith("skill"):
            opts["Skill Level"] = int(cfg[5:])
        e.configure(opts)
        _SF[cfg] = e
    return _SF[cfg]


def fermer_stockfish():
    for e in _SF.values():
        e.quit()
    _SF.clear()


def coup(joueur, board):
    if joueur.startswith("SF@"):
        return stockfish(joueur[3:]).play(board, chess.engine.Limit(time=SF_TEMPS)).move
    nom, rech, depth = decoder(joueur)
    b = board.copy(stack=True)
    if rech == "arbre":
        return choisir_arbre(b, evaluateur(nom), depth)
    # l'alpha-bêta évalue une position à la fois : sur Kaggle la latence CPU (~6 ms)
    # est deux fois plus basse que celle du GPU (~14 ms), donc on reste sur CPU
    return choisir_alphabeta(b, evaluateur(nom, gpu=False), depth)


# -- Stockfish -------------------------------------------------------------------------

URLS_STOCKFISH = [
    "https://github.com/official-stockfish/Stockfish/releases/download/sf_17.1/stockfish-ubuntu-x86-64-avx2.tar",
    "https://github.com/official-stockfish/Stockfish/releases/download/sf_17/stockfish-ubuntu-x86-64-avx2.tar",
    "https://github.com/official-stockfish/Stockfish/releases/download/sf_16.1/stockfish-ubuntu-x86-64-avx2.tar",
]


def installer_stockfish(dossier=None):
    dossier = dossier or os.path.join(os.path.dirname(os.path.abspath(DOSSIER_POIDS)), "stockfish")
    deja = glob.glob(os.path.join(dossier, "**", "stockfish-ubuntu-*"), recursive=True)
    deja = [d for d in deja if os.path.isfile(d) and not d.endswith(".tar")]
    if deja:
        return deja[0]
    os.makedirs(dossier, exist_ok=True)
    for url in URLS_STOCKFISH:
        try:
            tar = os.path.join(dossier, "sf.tar")
            urllib.request.urlretrieve(url, tar)
            with tarfile.open(tar) as t:
                t.extractall(dossier)
            binaire = [d for d in glob.glob(os.path.join(dossier, "**", "stockfish-ubuntu-*"), recursive=True)
                       if os.path.isfile(d) and not d.endswith(".tar")][0]
            os.chmod(binaire, os.stat(binaire).st_mode | stat.S_IEXEC)
            subprocess.run([binaire, "quit"], check=True, timeout=10)
            return binaire
        except Exception as e:
            print("échec", url, e)
    if shutil.which("stockfish") is None:
        subprocess.run("apt-get -qq install -y stockfish", shell=True)
    for p in (shutil.which("stockfish"), "/usr/games/stockfish"):
        if p and os.path.exists(p):
            return p
    raise RuntimeError("impossible d'installer Stockfish")


# -- puzzles Lichess ---------------------------------------------------------------

def resoudre_puzzle(joueur, fen, moves):
    """format Lichess : FEN avant le coup adverse, moves[0] = coup adverse, puis solution.
    Comme sur Lichess, un mat est toujours accepté même si ce n'est pas le coup attendu.
    renvoie (résolu, premier_coup_bon, nb_coups_joués, secondes)"""
    b = chess.Board(fen)
    moves = moves.split()
    b.push_uci(moves[0])
    premier, joues, t0 = None, 0, time.perf_counter()
    for i in range(1, len(moves), 2):
        attendu = chess.Move.from_uci(moves[i])
        mv = coup(joueur, b)
        joues += 1
        b.push(mv)
        mat = b.is_checkmate()
        b.pop()
        ok = mv == attendu or mat
        if premier is None:
            premier = ok
        if not ok:
            return False, premier, joues, time.perf_counter() - t0
        if mat:
            break
        b.push(attendu)
        if i + 1 < len(moves):
            b.push_uci(moves[i + 1])
    return True, premier, joues, time.perf_counter() - t0


# -- parties -------------------------------------------------------------------------

def jouer_partie(blancs, noirs, ouverture, max_plies=300):
    """ouverture = liste de coups UCI joués d'office ; renvoie un dict (résultat, pgn, temps)"""
    import chess.pgn
    b = chess.Board()
    for u in ouverture:
        b.push_uci(u)
    temps = {blancs: 0.0, noirs: 0.0}
    nb = {blancs: 0, noirs: 0}
    while not b.is_game_over(claim_draw=True) and b.ply() < max_plies:
        j = blancs if b.turn == chess.WHITE else noirs
        t0 = time.perf_counter()
        mv = coup(j, b)
        temps[j] += time.perf_counter() - t0
        nb[j] += 1
        b.push(mv)
    res = b.result(claim_draw=True) if b.is_game_over(claim_draw=True) else "1/2-1/2"
    fin = b.outcome(claim_draw=True)
    jeu = chess.pgn.Game.from_board(b)
    jeu.headers.update(Event="Benchmark P.A.W.N.", White=blancs, Black=noirs, Result=res)
    return {
        "blancs": blancs, "noirs": noirs, "resultat": res,
        "fin": fin.termination.name if fin else "MAX_PLIES",
        "plies": b.ply(), "pgn": str(jeu),
        "s_par_coup_blancs": temps[blancs] / max(1, nb[blancs]),
        "s_par_coup_noirs": temps[noirs] / max(1, nb[noirs]),
    }


# -- tâches pour ProcessPoolExecutor (doivent être importables depuis ce module) -------

def init_worker(gpu, threads, sf_chemin=None, sf_temps=0.1):
    configurer(gpu=gpu, threads=threads, sf_chemin=sf_chemin, sf_temps=sf_temps)


def tache_puzzle(args):
    joueur, pid, fen, moves = args
    ok, premier, joues, sec = resoudre_puzzle(joueur, fen, moves)
    return {"joueur": joueur, "PuzzleId": pid, "resolu": ok, "premier_coup": premier,
            "coups_joues": joues, "secondes": sec}


def tache_partie(args):
    blancs, noirs, ouverture, max_plies, id_ouv = args
    try:
        r = jouer_partie(blancs, noirs, ouverture, max_plies)
    finally:
        # un Stockfish ouvert garde un thread vivant qui empêche le processus fils de se terminer
        fermer_stockfish()
    r["ouverture"] = id_ouv
    return r

In [ ]:
import os, sys, time, json, random
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import chess
sys.path.insert(0, os.getcwd())  # pour que les processus fils trouvent commun_echecs.py
import onnxruntime as ort
import commun_echecs as ce

print("onnxruntime", ort.__version__, "| providers dispo :", ort.get_available_providers())
ev_test = ce.Evaluateur("PAWN")
print("session PAWN sur :", ev_test.session.get_providers())
print("éval position initiale :", ev_test.eval1(chess.STARTING_FEN), "cp")

In [ ]:
# ---- réglages ----
N_POSITIONS = 200_000   # positions de test pour les métriques d'éval
N_COUPS = 5_000         # positions pour l'accord de coup avec Stockfish
N_SYMETRIE = 20_000     # positions pour le test de symétrie
MIN_DEPTH = 20          # comme à l'entraînement
DEJA_VUS = 90_000_000   # PAWN : lignes 0-30 M, PAWN big : 0-60 M, SPAWN/contrôle : 60-90 M (cf. specs)
SEED = 0
GPU = True
SORTIE = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
rng = np.random.default_rng(SEED)

## 1. Positions de test (jamais vues à l'entraînement)

In [ ]:
from huggingface_hub import HfApi, HfFileSystem
import pyarrow.parquet as pq

for REPO in ["Lichess/chess-position-evaluations", "lichess/chess-evaluations"]:
    try:
        fichiers = sorted(f for f in HfApi().list_repo_files(REPO, repo_type="dataset") if f.endswith(".parquet"))
        if fichiers:
            break
    except Exception as e:
        print(REPO, "->", e)
print(REPO, ":", len(fichiers), "fichiers parquet")

fs = HfFileSystem()
nb_lignes = []
for f in fichiers:  # on ne lit que le pied de page de chaque parquet (quelques Ko)
    with fs.open(f"datasets/{REPO}/{f}") as h:
        nb_lignes.append(pq.ParquetFile(h).metadata.num_rows)
debuts = np.cumsum([0] + nb_lignes[:-1])
print(f"total : {sum(nb_lignes):,} lignes")

# on part du dernier fichier et on remonte tant qu'il faut, sans jamais descendre sous DEJA_VUS
choisis = []
for i in range(len(fichiers) - 1, -1, -1):
    if debuts[i] < DEJA_VUS:
        break
    choisis.append(i)
    if sum(nb_lignes[j] for j in choisis) > 6 * N_POSITIONS:
        break
assert choisis, "aucun fichier entièrement après les 90 M premières lignes !"
print("fichiers de test :", [fichiers[i] for i in choisis], "| première ligne :", f"{min(debuts[i] for i in choisis):,}")

from huggingface_hub import hf_hub_download
morceaux = []
for i in choisis:
    chemin = hf_hub_download(REPO, fichiers[i], repo_type="dataset")
    cols = [c for c in ["fen", "line", "depth", "cp", "mate"] if c in pq.ParquetFile(chemin).schema.names]
    morceaux.append(pd.read_parquet(chemin, columns=cols))
brut = pd.concat(morceaux, ignore_index=True)
print(brut.shape); brut.head()

In [ ]:
MATE_CP = 10_000
brut = brut[brut["depth"] >= MIN_DEPTH].copy()
# éval Lichess = point de vue des blancs ; mat -> ±(10000 - 10·n)
cp_blancs = brut["cp"].astype("float64")
m = brut["mate"].notna()
cp_blancs[m] = np.sign(brut.loc[m, "mate"]) * (MATE_CP - 10 * brut.loc[m, "mate"].abs())
brut["cp_blancs"] = cp_blancs
brut["trait_blanc"] = brut["fen"].str.split(" ").str[1] == "w"
brut["cp_trait"] = np.where(brut["trait_blanc"], brut["cp_blancs"], -brut["cp_blancs"])

# plusieurs lignes par position (multi-PV, plusieurs profondeurs) : la plus profonde, puis la meilleure pour le camp au trait
fens_uniques = brut["fen"].unique()
tirage = set(rng.choice(fens_uniques, size=min(N_POSITIONS, len(fens_uniques)), replace=False))
df = brut[brut["fen"].isin(tirage)].sort_values(["fen", "depth", "cp_trait"], ascending=[True, False, False])
df = df.groupby("fen", as_index=False).first()
del brut
# les FEN du dataset n'ont pas toujours les compteurs de coups
df["fen"] = df["fen"].apply(lambda f: f if len(f.split()) == 6 else f + " 0 1")
print(len(df), "positions de test"); df.head()

In [ ]:
VALEUR = {chess.KNIGHT: 3, chess.BISHOP: 3, chess.ROOK: 5, chess.QUEEN: 9}
def phase(fen):
    b = chess.Board(fen)
    mat = sum(VALEUR.get(p.piece_type, 0) for p in b.piece_map().values())
    return "ouverture" if mat >= 50 else ("milieu" if mat >= 20 else "finale")
df["phase"] = df["fen"].map(phase)
df["zone"] = pd.cut(df["cp_trait"].abs(), [-1, 50, 150, 400, 1000, 1e9],
                    labels=["égal <50", "léger 50-150", "net 150-400", "gagnant 400-1000", "décisif/mat"])
df[["phase", "zone"]].value_counts().unstack()

## 2. Évaluation des 4 réseaux

In [ ]:
def wp(cp):
    return 1 / (1 + np.exp(-np.asarray(cp, dtype=np.float64) / 400))

cible = df["cp_trait"].to_numpy()
p_cible = wp(cible)
fens = df["fen"].tolist()
preds = {}
for nom in ce.RESEAUX:
    ev = ce.Evaluateur(nom, gpu=GPU)
    t0 = time.perf_counter()
    preds[nom] = ev.evals(fens, lot=8192).astype(np.float64)
    print(f"{nom:9s} {len(fens) / (time.perf_counter() - t0):>10,.0f} positions/s  ({ev.session.get_providers()[0]})")
    df["pred_" + nom] = preds[nom]

In [ ]:
from scipy.stats import spearmanr, pearsonr

def metriques(cible, pred):
    p, q = wp(cible), np.clip(wp(pred), 1e-7, 1 - 1e-7)
    c1, c2 = np.clip(cible, -1000, 1000), np.clip(pred, -1000, 1000)
    net = np.abs(cible) >= 100
    return {
        "BCE": float(np.mean(-(p * np.log(q) + (1 - p) * np.log(1 - q)))),
        "MAE win-prob (pts %)": float(100 * np.mean(np.abs(p - q))),
        "MAE cp (±1000)": float(np.mean(np.abs(c1 - c2))),
        "Pearson cp": float(pearsonr(c1, c2)[0]),
        "Spearman": float(spearmanr(cible, pred)[0]),
        "bon camp (%)": float(100 * np.mean(np.sign(cible[net]) == np.sign(pred[net]))),
    }

tab = pd.DataFrame({nom: metriques(cible, preds[nom]) for nom in preds}).T
# référence : BCE minimale atteignable (entropie de la cible) et un modèle « toujours 0 cp »
h = float(np.mean(-(p_cible * np.log(np.clip(p_cible, 1e-12, 1)) + (1 - p_cible) * np.log(np.clip(1 - p_cible, 1e-12, 1)))))
tab.loc["(réf.) toujours 0 cp"] = metriques(cible, np.zeros_like(cible))
print(f"entropie de la cible (BCE parfaite) = {h:.4f}")
tab.style.format("{:.4f}").background_gradient(axis=0, cmap="RdYlGn_r", subset=["BCE", "MAE win-prob (pts %)", "MAE cp (±1000)"])

In [ ]:
lignes = []
for groupe in ["phase", "zone"]:
    for val, sous in df.groupby(groupe, observed=True):
        for nom in preds:
            m = metriques(sous["cp_trait"].to_numpy(), sous["pred_" + nom].to_numpy())
            lignes.append({"groupe": groupe, "valeur": val, "modèle": nom, "n": len(sous), **m})
detail = pd.DataFrame(lignes)
detail.pivot_table(index=["groupe", "valeur", "n"], columns="modèle", values="MAE win-prob (pts %)").round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
bins = np.linspace(0, 1, 21)
centre = (bins[:-1] + bins[1:]) / 2
for nom in preds:
    q = wp(preds[nom]); idx = np.digitize(q, bins) - 1
    moy = [p_cible[idx == k].mean() if (idx == k).sum() > 50 else np.nan for k in range(20)]
    axes[0].plot(centre, moy, marker="o", label=nom)
axes[0].plot([0, 1], [0, 1], "k--", lw=1)
axes[0].set(xlabel="win-prob prédite", ylabel="win-prob Stockfish (moyenne)", title="Calibration")
axes[0].legend()
pv = detail[detail.groupe == "phase"].pivot(index="valeur", columns="modèle", values="MAE win-prob (pts %)")
pv.plot.bar(ax=axes[1], rot=0, title="MAE win-prob par phase (plus bas = mieux)")
plt.tight_layout(); plt.savefig(f"{SORTIE}/eval_calibration.png", dpi=120); plt.show()

## 3. Accord avec le meilleur coup de Stockfish (profondeur 1, sans recherche)
Pour chaque position on évalue tous les coups légaux avec le réseau et on garde le meilleur ;
on compare avec le premier coup de la ligne principale de Stockfish.

In [ ]:
sous = df[df["line"].notna() & (df["line"].str.len() > 0)].sample(min(N_COUPS, len(df)), random_state=SEED)
enfants, index, terminaux, coups_sf, legaux = [], [], {}, [], []
for k, (fen, ligne) in enumerate(zip(sous["fen"], sous["line"])):
    b = chess.Board(fen)
    coups = list(b.legal_moves)
    legaux.append(coups)
    coups_sf.append(chess.Move.from_uci(ligne.split()[0]))
    for j, mv in enumerate(coups):
        b.push(mv)
        if b.is_checkmate():
            terminaux[(k, j)] = ce.MATE
        elif b.is_game_over():
            terminaux[(k, j)] = 0
        else:
            index.append((k, j)); enfants.append(b.fen())
        b.pop()
print(len(sous), "positions,", len(enfants), "enfants à évaluer")

accord = {}
for nom in ce.RESEAUX:
    v = ce.Evaluateur(nom, gpu=GPU).evals(enfants, lot=8192)
    scores = [np.full(len(c), -np.inf) for c in legaux]
    for (k, j), x in zip(index, v):
        scores[k][j] = -x
    for (k, j), x in terminaux.items():
        scores[k][j] = x
    top1 = top3 = 0
    for k, s in enumerate(scores):
        ordre = [legaux[k][j] for j in np.argsort(-s)]
        top1 += ordre[0] == coups_sf[k]
        top3 += coups_sf[k] in ordre[:3]
    accord[nom] = {"top-1 (%)": 100 * top1 / len(scores), "top-3 (%)": 100 * top3 / len(scores)}
accord = pd.DataFrame(accord).T
accord

## 4. Symétrie gauche ↔ droite
Sans droits de roque, une position et son miroir (colonne a ↔ colonne h) ont exactement la même valeur.
Un bon évaluateur doit donner presque la même note aux deux.

In [ ]:
sans_roque = df[df["fen"].str.split(" ").str[2] == "-"].head(N_SYMETRIE)
miroirs = [chess.Board(f).transform(chess.flip_horizontal).fen() for f in sans_roque["fen"]]
sym = {}
for nom in ce.RESEAUX:
    a = sans_roque["pred_" + nom].to_numpy()
    b = ce.Evaluateur(nom, gpu=GPU).evals(miroirs, lot=8192)
    d = np.abs(a - b)
    sym[nom] = {"écart moyen (cp)": d.mean(), "écart médian (cp)": np.median(d), "% écart > 50 cp": 100 * (d > 50).mean()}
sym = pd.DataFrame(sym).T
print(len(sans_roque), "positions"); sym

## 5. Vitesse et taille

In [ ]:
PARAMS_SPEC = {"PAWN": 1_414_906, "PAWN_BIG": 3_638_178, "CONTROLE": 3_638_178, "SPAWN": 3_649_698}
lot = fens[:4096]
vitesse = {}
for nom in ce.RESEAUX:
    ligne = {"paramètres": PARAMS_SPEC[nom], "fichier (Mo)": os.path.getsize(ce.poids(ce.RESEAUX[nom])) / 1e6}
    for dev in (["GPU", "CPU"] if "CUDAExecutionProvider" in ort.get_available_providers() else ["CPU"]):
        ev = ce.Evaluateur(nom, gpu=dev == "GPU")
        ev.evals(lot[:64])  # chauffe
        t = []
        for f in lot[:200]:
            t0 = time.perf_counter(); ev.evals([f]); t.append(time.perf_counter() - t0)
        t0 = time.perf_counter(); ev.evals(lot, lot=4096); debit = len(lot) / (time.perf_counter() - t0)
        ligne[f"latence {dev} (ms)"] = 1000 * np.median(t)
        ligne[f"débit {dev} (pos/s)"] = debit
    vitesse[nom] = ligne
vitesse = pd.DataFrame(vitesse).T
vitesse.round(2)

## 6. Synthèse

In [ ]:
synthese = tab.drop(index="(réf.) toujours 0 cp").join(accord).join(sym).join(vitesse)
synthese.to_csv(f"{SORTIE}/bench1_evaluation.csv")
detail.to_csv(f"{SORTIE}/bench1_detail_phase_zone.csv", index=False)
cols = ["BCE", "MAE win-prob (pts %)", "Spearman", "bon camp (%)", "top-1 (%)", "top-3 (%)", "écart moyen (cp)"]
fig, axes = plt.subplots(1, len(cols), figsize=(3.2 * len(cols), 3.5))
for ax, c in zip(axes, cols):
    synthese[c].plot.bar(ax=ax, color=["#9aa", "#58a", "#a85", "#5a5"]); ax.set_title(c, fontsize=9); ax.tick_params(labelsize=8)
plt.tight_layout(); plt.savefig(f"{SORTIE}/bench1_synthese.png", dpi=120); plt.show()
synthese[cols].round(3)

### Comment lire les résultats
- **CONTROLE vs SPAWN** est la comparaison clé : même base, mêmes données, même ordre → la différence vient **uniquement du module Soluce**.
- **PAWN_BIG vs CONTROLE** mesure l'effet de « juste plus d'entraînement ».
- Si **top-1** reste bas (< 40 %) alors que la BCE est bonne, c'est normal : l'éval statique ne voit pas les tactiques, c'est le rôle de la recherche (→ notebooks 2 et 3).